In [1]:
import os
import torch
import onnx
from omegaconf import OmegaConf

from models import utils
from models.src.yolo import YOLO

/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BATCH_SIZE = 1
experiment_root = "experiments"
trained_model_root = "trained_model"
sample_image = "samples/0000310_03000_d_0000124.jpg" #"samples/1f0e643b125f00ec.jpg"
model_name = "tough_river"
model_tag = "best_map"

In [3]:
cfg = utils.load_model_yaml(model_name, experiment_root)
model_path = utils.find_model_pth_paths(trained_model_root, model_name, model_tag)
cfg.model.metadata.best_model_folder = os.path.dirname(model_path[0])
if OmegaConf.select(cfg, "model.attention_type") is not None:
    attention_type = cfg.model.attention_type
    print("attention_type:", attention_type)
else:
    print("attention_type does not exist")
    cfg.model.attention_type = None
inference_model_name = os.path.splitext(os.path.basename(model_path[0]))[0]

attention_type does not exist


In [4]:
model = YOLO(backbone_name=cfg.model.backbone_name,
                     train_backbone=cfg.model.train_backbone,
                     returned_layers=cfg.model.returned_layers,
                     num_classes=cfg.dataset.nc,
                     fpn_channels=cfg.model.fpn_channels,
                     attention_type=cfg.model.attention_type,)

/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Freezing conv1.weight
Freezing bn1.weight
Freezing bn1.bias
Freezing layer1.0.conv1.weight
Freezing layer1.0.bn1.weight
Freezing layer1.0.bn1.bias
Freezing layer1.0.conv2.weight
Freezing layer1.0.bn2.weight
Freezing layer1.0.bn2.bias
Freezing layer1.1.conv1.weight
Freezing layer1.1.bn1.weight
Freezing layer1.1.bn1.bias
Freezing layer1.1.conv2.weight
Freezing layer1.1.bn2.weight
Freezing layer1.1.bn2.bias
Freezing layer1.2.conv1.weight
Freezing layer1.2.bn1.weight
Freezing layer1.2.bn1.bias
Freezing layer1.2.conv2.weight
Freezing layer1.2.bn2.weight
Freezing layer1.2.bn2.bias
Freezing fc.weight
Freezing fc.bias


In [5]:
model, _ = utils.load_model(model, cfg.model.metadata.best_model_folder, model_name=inference_model_name)

In [6]:
cfg.model.metadata.best_model_folder

'/Users/albertomarengo/Bitstrapped-Project/my_yolo/my_yolo/trained_model/tough_river/best_map'

In [7]:
inference_model_name

'YOLO_train'

In [8]:
onnx_path = cfg.model.metadata.best_model_folder + '/' + inference_model_name + '.onnx'

In [9]:
cfg.model

{'allow_multi_level': True, 'attention_type': None, 'backbone_name': 'resnet34', 'center_radius': 1.5, 'fpn_channels': 256, 'image_size': [640, 640, 3], 'max_detections': 100, 'metadata': {'best_model_folder': '/Users/albertomarengo/Bitstrapped-Project/my_yolo/my_yolo/trained_model/tough_river/best_map'}, 'name': 'yolo', 'nms_threshold': 0.5, 'returned_layers': [1, 2, 3, 4], 'score_threshold': 0.01, 'top_k_per_level': 3, 'train_backbone': True}

In [10]:
onnx_path

'/Users/albertomarengo/Bitstrapped-Project/my_yolo/my_yolo/trained_model/tough_river/best_map/YOLO_train.onnx'

In [11]:
model.eval()

dummy = torch.randn(BATCH_SIZE, 3, 640, 640)

torch.onnx.export(
    model,
    dummy,
    onnx_path,
    input_names=["inputs"],
    output_names=["x"],
    opset_version=17,
    do_constant_folding=True,
)

W0613 16:17:33.347000 84612 yol-venv/lib/python3.12/site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `YOLO([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `YOLO([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/albertomarengo/Bitstrapped-Project/my_yolo/yol-venv/lib/python3.12/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = 

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.11.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"inputs"<FLOAT,[1,3,640,640]>
            ),
            outputs=(
                %"x"<FLOAT,[1,34000,15]>
            ),
            initializers=(
                %"neck.fpn_layer_1x1.bias"<FLOAT,[256]>{TorchTensor(...)},
                %"neck.pan_last_layer_3x3.bias"<FLOAT,[256]>{TorchTensor(...)},
                %"neck.fpn_layers_lat_conns.0.conv1x1.bias"<FLOAT,[256]>{TorchTensor(...)},
                %"neck.fpn_layers_lat_conns.1.conv1x1.bias"<FLOAT,[256]>{TorchTensor(...)},
                %"neck.fpn_layers_lat_conns.2.conv1x1.bias"<FLOAT,[256]>{TorchTensor(...)},
                %"neck.fpn_layers_3x3.0.bias"<FLOAT,[256]>{TorchTensor(...)},
                %"n

In [12]:
import onnxruntime as ort
import numpy as np

available_providers = ort.get_available_providers()
print("Available providers:", available_providers)

providers = []

if "CUDAExecutionProvider" in available_providers:
    providers.append("CUDAExecutionProvider")

providers.append("CPUExecutionProvider")

session = ort.InferenceSession(
    onnx_path,
    providers=providers,
)

input_name = session.get_inputs()[0].name

image = np.random.randn(BATCH_SIZE, 3, 640, 640).astype(np.float32)

predictions = session.run(None, {input_name: image})[0]

print(predictions.shape)

Available providers: ['CoreMLExecutionProvider', 'AzureExecutionProvider', 'CPUExecutionProvider']
(1, 34000, 15)


In [13]:
tensor_image = torch.from_numpy(image).float()

In [14]:
pred = model(tensor_image)

In [15]:
import time
start = time.perf_counter()
predictions = session.run(None, {input_name: image})[0]
end = time.perf_counter()

print(end - start)

start = time.perf_counter()
pred = model(tensor_image)
end = time.perf_counter()

print(end - start)

0.960695166606456
0.7359036672860384


In [16]:
import torch
import numpy as np
import onnxruntime as ort

def benchmark_torch(model, x, n_warmup=10, n_runs=50):
    model.eval()
    model.cpu()

    with torch.inference_mode():
        for _ in range(n_warmup):
            _ = model(x)

    start = time.perf_counter()

    with torch.inference_mode():
        for _ in range(n_runs):
            _ = model(x)

    end = time.perf_counter()
    return (end - start) / n_runs


def benchmark_onnx(session, x_np, n_warmup=10, n_runs=50):
    input_name = session.get_inputs()[0].name

    for _ in range(n_warmup):
        _ = session.run(None, {input_name: x_np})

    start = time.perf_counter()

    for _ in range(n_runs):
        _ = session.run(None, {input_name: x_np})

    end = time.perf_counter()
    return (end - start) / n_runs


x = torch.randn(32, 3, 640, 640).float()
x_np = x.numpy().astype(np.float32)

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession(
    onnx_path,
    sess_options=sess_options,
    providers=["CPUExecutionProvider"],
)

# torch_s = benchmark_torch(model, x)
# onnx_s = benchmark_onnx(session, x_np)
#
# print(f"PyTorch: {torch_s:.4f} s / batch")
# print(f"ONNX:    {onnx_s:.4f} s / batch")
# print(f"Speedup: {torch_s / onnx_s:.3f}x")

In [19]:
import openvino as ov
import numpy as np

core = ov.Core()

compiled_model = core.compile_model(
    onnx_path,
    device_name="CPU",
)

input_layer = compiled_model.input(0)

x_np = np.random.randn(BATCH_SIZE, 3, 640, 640).astype(np.float32)

start = time.perf_counter()
_ = compiled_model({input_layer: x_np})
end = time.perf_counter()

print(end - start)

for _ in range(10):
    _ = compiled_model({input_layer: x_np})

start = time.perf_counter()

for _ in range(100):
    _ = compiled_model({input_layer: x_np})

end = time.perf_counter()

print("OpenVINO ms:", (end - start) / 100 * 1000)

0.3681921251118183
OpenVINO ms: 190.51873167045414
